# 03 - Tracking and team classification

Pipeline on a test video: YOLO26n detection (1280 px) -> BoT-SORT tracking -> per-track role vote -> team clustering on jersey color.

Inputs (Google Drive, `football-analytics/`): `runs/yolo26n_1280/weights/best.pt` and a video in `videos/input/`.

In [ ]:
!nvidia-smi
!pip install -q ultralytics==8.4.174

## 1. Setup

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

base = Path("/content/drive/MyDrive/football-analytics")
CLIP = "clip01.mp4"

src = base / "videos/input" / CLIP
out = base / "videos/output" / f"{src.stem}_botsort"
weights = base / "runs/yolo26n_1280/weights/best.pt"
assert src.exists(), f"missing video: {src}"
assert weights.exists(), f"missing weights: {weights}"

In [ ]:
import yaml
import ultralytics

cfg = yaml.safe_load((Path(ultralytics.__file__).parent / "cfg/trackers/botsort.yaml").read_text())
cfg["track_buffer"] = 90

tracker_cfg = "/content/botsort_buf90.yaml"
with open(tracker_cfg, "w") as f:
    yaml.dump(cfg, f)

## 2. Tracking

In [ ]:
import pandas as pd
from ultralytics import YOLO

model = YOLO(weights)

rows = []
for i, r in enumerate(model.track(
    source=str(src),
    imgsz=1280,
    conf=0.10,
    classes=[1, 2, 3],
    tracker=tracker_cfg,
    agnostic_nms=True,
    stream=True,
    verbose=False,
    save=True,
    project=str(out.parent),
    name=out.name,
    exist_ok=True,
)):
    if r.boxes.id is None:
        continue
    for box, tid, c, p in zip(r.boxes.xyxy.tolist(), r.boxes.id.int().tolist(), r.boxes.cls.int().tolist(), r.boxes.conf.tolist()):
        rows.append({"frame": i, "track_id": tid, "cls": c, "conf": p, "x1": box[0], "y1": box[1], "x2": box[2], "y2": box[3]})

tracks = pd.DataFrame(rows)
tracks.to_csv(out / "tracks.csv", index=False)
print(tracks.shape, "| tracks:", tracks.track_id.nunique())

In [ ]:
!ffmpeg -y -loglevel error -i "{out}/{src.stem}.avi" -c:v libx264 -crf 23 -pix_fmt yuv420p "{out}/{src.stem}_tracked.mp4"
!ls -lh "{out}"

## 3. Track cleanup and role vote

In [ ]:
cls_map = {1: "goalkeeper", 2: "player", 3: "referee"}


def clean_tracks(df, min_len=15):
    length = df.groupby("track_id").frame.transform("size")
    df = df[length >= min_len].copy()
    votes = df.groupby(["track_id", "cls"]).conf.sum().reset_index()
    best = votes.loc[votes.groupby("track_id").conf.idxmax()].set_index("track_id").cls
    df["cls_smooth"] = df.track_id.map(best)
    return df


tracks_clean = clean_tracks(tracks)
print("detections:", len(tracks), "->", len(tracks_clean))
print("tracks:", tracks.track_id.nunique(), "->", tracks_clean.track_id.nunique())
print(tracks_clean.groupby("track_id").cls_smooth.first().map(cls_map).value_counts())
print(tracks_clean.groupby("frame").size().describe())

## 4. Team classification

In [ ]:
import cv2
import numpy as np

players = tracks_clean[tracks_clean.cls_smooth == 2]
sample = (
    players[(players.y2 - players.y1) >= 40]
    .sort_values("conf", ascending=False)
    .groupby("track_id")
    .head(8)
)
needed = {f: g for f, g in sample.groupby("frame")}

cap = cv2.VideoCapture(str(src))
feats, crops = [], []
i = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if i in needed:
        for r in needed[i].itertuples():
            x1, y1, x2, y2 = (max(int(v), 0) for v in (r.x1, r.y1, r.x2, r.y2))
            h, w = y2 - y1, x2 - x1
            crop = frame[y1 + int(0.15 * h):y1 + int(0.55 * h), x1 + int(0.2 * w):x2 - int(0.2 * w)]
            if crop.size == 0:
                continue
            hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
            grass = (hsv[..., 0] >= 35) & (hsv[..., 0] <= 85) & (hsv[..., 1] > 40)
            if (~grass).sum() < 20:
                continue
            lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB)[~grass]
            feats.append([r.track_id, *np.median(lab, axis=0)])
            crops.append((r.track_id, crop))
    i += 1
cap.release()

track_feat = pd.DataFrame(feats, columns=["track_id", "L", "a", "b"]).groupby("track_id").median()
print("player tracks with a usable crop:", len(track_feat), "/", players.track_id.nunique())

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(track_feat.values)
team = pd.Series(km.labels_, index=track_feat.index, name="team")
print(team.value_counts())

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for ax in axes.ravel():
    ax.axis("off")
for t in (0, 1):
    for ax, tid in zip(axes[t], team[team == t].index[:8]):
        crop = next(c for k, c in crops if k == tid)
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        ax.set_title(f"team {t} | track {tid}")
plt.tight_layout()

In [ ]:
tracks_clean["team"] = tracks_clean.track_id.map(team)
tracks_clean.to_csv(out / "tracks_teams.csv", index=False)
print(tracks_clean.groupby("cls_smooth").team.value_counts(dropna=False))

## 5. Ball detection and trajectory

In [ ]:
import cv2
import pandas as pd
from ultralytics import YOLO

ball_model = YOLO(weights)

ball_rows = []
for i, r in enumerate(ball_model.predict(
    source=str(src),
    imgsz=1280,
    conf=0.05,
    classes=[0],
    stream=True,
    verbose=False,
)):
    for box, p in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist()):
        ball_rows.append({"frame": i, "conf": p, "x1": box[0], "y1": box[1], "x2": box[2], "y2": box[3]})

balls = pd.DataFrame(ball_rows)
balls.to_csv(out / "ball_candidates.csv", index=False)

n_frames = int(cv2.VideoCapture(str(src)).get(cv2.CAP_PROP_FRAME_COUNT))
best = balls.sort_values("conf", ascending=False).groupby("frame").head(1)

print("candidates:", len(balls), "| frames:", n_frames)
for thr in (0.05, 0.10, 0.25, 0.50):
    print(f"best conf >= {thr}: {(best.conf >= thr).sum()} / {n_frames} frames")
print("candidates per frame:", balls.groupby("frame").size().describe().round(1).to_dict())
print("median ball box (w, h):", round(float((balls.x2 - balls.x1).median()), 1), round(float((balls.y2 - balls.y1).median()), 1))

In [ ]:
import cv2
import numpy as np
import pandas as pd

balls = pd.read_csv(out / "ball_candidates.csv")
balls["cx"] = (balls.x1 + balls.x2) / 2
balls["cy"] = (balls.y1 + balls.y2) / 2
balls = balls.sort_values("frame").reset_index(drop=True)

T = int(cv2.VideoCapture(str(src)).get(cv2.CAP_PROP_FRAME_COUNT))
MISS, GAP, V_MAX, V_SCALE = 2.0, 12, 150.0, 30.0

fr = balls.frame.to_numpy()
xy = balls[["cx", "cy"]].to_numpy()
node = -np.log(balls.conf.to_numpy() + 1e-3)

best = np.empty(len(balls))
prev = np.full(len(balls), -1)
lo = 0
for i in range(len(balls)):
    best[i] = node[i] + MISS * fr[i]
    while fr[lo] < fr[i] - GAP:
        lo += 1
    for j in range(lo, i):
        dt = fr[i] - fr[j]
        if dt < 1:
            continue
        v = np.hypot(*(xy[i] - xy[j])) / dt
        if v > V_MAX:
            continue
        c = best[j] + node[i] + v / V_SCALE + MISS * (dt - 1)
        if c < best[i]:
            best[i], prev[i] = c, j

i = int(np.argmin(best + MISS * (T - 1 - fr)))
path = []
while i != -1:
    path.append(i)
    i = prev[i]
path = path[::-1]

sel = balls.loc[path, ["frame", "cx", "cy", "conf"]].set_index("frame")
traj = sel.reindex(range(T))
traj["interpolated"] = traj.cx.isna() & (traj.index >= sel.index.min()) & (traj.index <= sel.index.max())
traj[["cx", "cy"]] = traj[["cx", "cy"]].interpolate(limit_area="inside")
traj.to_csv(out / "ball_track.csv")

print("frames:", T, "| candidates:", len(balls), "| chosen:", len(path))
print("frames with a ball position:", int(traj.cx.notna().sum()), "| interpolated:", int(traj.interpolated.sum()))
print("chosen conf median:", round(float(sel.conf.median()), 2), "| chosen with conf < 0.25:", int((sel.conf < 0.25).sum()))
print("first / last chosen frame:", int(sel.index.min()), int(sel.index.max()))

## 6. Output video

In [ ]:
import cv2
import numpy as np
import pandas as pd

tt = pd.read_csv(out / "tracks_teams.csv")
traj = pd.read_csv(out / "ball_track.csv", index_col=0)
gk_share = tt.groupby("track_id").cls.apply(lambda s: (s == 1).mean())
gk_tracks = set(gk_share[gk_share >= 0.15].index)

colors = {"team0": (0, 0, 255), "team1": (255, 120, 0), "referee": (0, 255, 255), "gk": (0, 200, 0), "other": (160, 160, 160)}
BALL = (255, 0, 255)
by_frame = {f: g for f, g in tt.groupby("frame")}

cap = cv2.VideoCapture(str(src))
fps = cap.get(cv2.CAP_PROP_FPS)
size = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
raw_path = out / f"{src.stem}_full_raw.mp4"
writer = cv2.VideoWriter(str(raw_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, size)

trail = []
i = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    g = by_frame.get(i)
    if g is not None:
        for r in g.itertuples():
            if r.track_id in gk_tracks:
                key, tag = "gk", "GK?"
            elif r.cls_smooth == 3:
                key, tag = "referee", "REF"
            elif r.team == 0:
                key, tag = "team0", "T0"
            elif r.team == 1:
                key, tag = "team1", "T1"
            else:
                key, tag = "other", ""
            p1, p2 = (int(r.x1), int(r.y1)), (int(r.x2), int(r.y2))
            cv2.rectangle(frame, p1, p2, colors[key], 2)
            cv2.putText(frame, f"{tag} {r.track_id}", (p1[0], max(p1[1] - 6, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, colors[key], 2)
    if i in traj.index and pd.notna(traj.at[i, "cx"]):
        pt = (int(traj.at[i, "cx"]), int(traj.at[i, "cy"]))
        trail = (trail + [pt])[-20:]
        for a, b in zip(trail[:-1], trail[1:]):
            cv2.line(frame, a, b, BALL, 2)
        cv2.circle(frame, pt, 10, BALL, 2 if traj.at[i, "interpolated"] else -1)
    else:
        trail = []
    writer.write(frame)
    i += 1
cap.release()
writer.release()

final_path = out / f"{src.stem}_full.mp4"
!ffmpeg -y -loglevel error -i "{raw_path}" -c:v libx264 -crf 23 -pix_fmt yuv420p "{final_path}"
!ls -lh "{out}"